# Gemini Human Recognize

In [120]:
%pip install --upgrade --quiet google-genai pillow dotenv pandas

Note: you may need to restart the kernel to use updated packages.


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()

True

In [123]:
PROJECT_ID = os.environ['PROJECT_ID']
LOCATION = os.environ['LOCATION']
GS_BUCKET = os.environ['GS_BUCKET']
BUCKET = os.environ['BUCKET']

In [7]:
MODEL_ID= "gemini-2.0-flash-001"

In [13]:
from google import genai
from google.genai import types
from google.genai.types import (
    GenerateContentConfig,
    GoogleSearch,
    Part,
    Tool,
)
from IPython.display import HTML, Markdown, display

In [37]:
client = genai.Client(vertexai=True, project=PROJECT_ID, location=LOCATION)

In [103]:
def detect_with_gemini(image_path):
    #system_instruction="You are an image analysis AI. You can identify people in images and provide their names"
    system_instruction="당신은 이미지 분석하는 일을 담당합니다. 주어진 이미지에서 인물을 찾아서 인물의 이름을 알려주세요. 인물이 누구인지 모르거나, 100% 신뢰할 수 없다면 '모름'으로 답하세요."

    google_search_tool = Tool(google_search=GoogleSearch())

    grounding_config = GenerateContentConfig(
        temperature = 0.1,
        top_p = 0.99,
        max_output_tokens = 8192,
        response_modalities = ["TEXT"],
        system_instruction=system_instruction,
        tools=[google_search_tool]   
    )

    #prompt = f"""If you can identify the people in the image, provide their names ONLY. If you don't know them, Don't make up names and just respond with 'unknown'"""
    prompt = f"""설명을 붙이지 말고, 이름만 대답하세요."""

    with open(image_path, "rb") as f:
        image = f.read()

    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[Part.from_bytes(data=image, mime_type="image/png"),
            prompt,],
        config=grounding_config
    )

    print(response.text)


In [143]:
def detect_with_gemini_url(image_path, percent, grounding=True):
    #system_instruction="You are an image analysis AI. You can identify people in images and provide their names"
    system_instruction=f"당신은 이미지 분석하는 일을 담당합니다. 주어진 이미지에서 인물을 찾아서 인물의 이름을 알려주세요. 인물이 누구인지 모르거나, {percent}% 신뢰할 수 없다면 '모름'으로 답하세요."

    google_search_tool = Tool(google_search=GoogleSearch())

    grounding_config = GenerateContentConfig(
        temperature = 0.1,
        top_p = 0.99,
        max_output_tokens = 8192,
        response_modalities = ["TEXT"],
        system_instruction=system_instruction,
        tools=[google_search_tool]   
    )

    if grounding:
        grounding_config = GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.99,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            system_instruction=system_instruction,
            tools=[google_search_tool]   
        )
    else:
        grounding_config = GenerateContentConfig(
            temperature = 0.1,
            top_p = 0.99,
            max_output_tokens = 8192,
            response_modalities = ["TEXT"],
            system_instruction=system_instruction,  
        )

    #prompt = f"""If you can identify the people in the image, provide their names ONLY. If you don't know them, Don't make up names and just respond with 'unknown'"""
    prompt = f"""설명을 붙이지 말고, 이름만 대답하세요."""


    response = client.models.generate_content(
        model=MODEL_ID,
        contents=[Part.from_uri(file_uri=f"{GS_BUCKET}/human_images/{image_path}", mime_type="image/jpeg"),
            prompt,],
        config=grounding_config
    )

    print(response.text)
    return response.text

## Vision

In [ ]:
%pip install google-cloud-vision

In [73]:
from google.cloud import vision

In [62]:
vision_client = vision.ImageAnnotatorClient()

In [63]:
def detect_with_vision(image_path):
    with open(image_path, "rb") as f:
        image = f.read()
    vision_image = vision.Image(content=image)
    web_detection = vision_client.web_detection(image=vision_image).web_detection
    results = [d.description for d in web_detection.web_entities]
    print(results)

<img src=./human1.png width="10%">

In [104]:
detect_with_gemini("human1.png")

모름


In [88]:
detect_with_vision("human1.png")

['Yoon Suk Yeol', 'Myung Tae-kyun', 'Newstapa', '2024 South Korean martial law crisis', 'Journalism', 'People Power Party', 'Investigative journalism', 'Korea exploration Journalism Center', '주간뉴스타파', 'Reporter']


<img src=./human3.png width="10%">

In [105]:
detect_with_gemini("human3.png")

모름


In [92]:
detect_with_vision("human3.png")

['Lee Yi-kyung', 'Song Eun-yi', 'Kwon Il-yong', 'Brave Detectives', 'Detective', 'Criminal investigation', 'Television', 'MyDramaList', 'Crime', 'Netflix']


<img src=./human5.png width="10%">

In [106]:
detect_with_gemini("human5.png")

모름


In [94]:
detect_with_vision("human5.png")

['Kim Bu-gyeom', 'Politics', 'Minister', 'Member of the National Assembly of the Republic of Korea', 'General election', '2020 South Korean legislative election', 'Election law', 'Legislative elections in South Korea']


<img src="./human4.png" width="10%">

In [107]:
detect_with_gemini("human4.png")

모름



In [96]:
detect_with_vision("human4.png")

['Lee Yeon-bok', 'Please Take Care of My Refrigerator', 'Kim Seong-joo', '냉장고를 부탁해 2', 'JTBC', 'Variety show', 'Refrigerator', 'Television', '2014', 'MyDramaList']


<img src=./human6.png width="10%">

In [108]:
detect_with_gemini("human6.png")

이선균, 아이유



In [98]:
detect_with_vision("human6.png")

['IU', 'Lee Sun-kyun', 'My Mister', 'Times Square, Seoul', '데일리안', 'tvN', 'Actor', 'Photograph', 'Laughter', 'tvN DRAMA']


In [140]:
import pandas as pd
from IPython.display import display, HTML

def display_dataframe_with_images(df):

    html = df.copy()
    html = html.to_html(escape=False, index=False)

    display(HTML(html))

In [ ]:
df = pd.DataFrame(columns=['image', 'name_100%', 'name_90%', 'name_80%'])

for i in range(1,69):
    names_100 = detect_with_gemini_url(f"image_{i}.jpeg", 100)
    names_90 = detect_with_gemini_url(f"image_{i}.jpeg", 90)
    names_80 = detect_with_gemini_url(f"image_{i}.jpeg", 80)
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'image': [image_data], 'name_100%': [names_100], 'name_90%': [names_90], 'name_80%': [names_80]})
    df = pd.concat([df, new_row], ignore_index=True)


In [142]:
display_dataframe_with_images(df)

image,name_100%,name_90%,name_80%
,성룡\n,성룡\n,성룡\n
,윤여정\n,윤여정\n,윤여정\n
,레오나르도 디카프리오\n,레오나르도 디카프리오\n,레오나르도 디카프리오\n
,양세형\n,제이쓴\n,모름
,김수현\n,김수현\n,김수현\n
,스티브 잡스\n,스티브 잡스\n,스티브 잡스\n
,모름\n,모름\n,모름\n
,"다음은 이미지에 있는 인물의 이름입니다.\n\n신하균, 김고은\n","조승우, 송지효\n","조승우, 송지효\n"
,박원순\n,박원순\n,박원순\n
,문재인\n,문재인\n,문재인\n


## Without Grounding

In [ ]:
df_2 = pd.DataFrame(columns=['image', 'name_100%'])

for i in range(1,69):
    names_100 = detect_with_gemini_url(f"image_{i}.jpeg", 100, False)
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'image': [image_data], 'name_100%': [names_100]})
    df_2 = pd.concat([df_2, new_row], ignore_index=True)

성룡
윤여정
레오나르도 디카프리오
홍현희
김수현
스티브 잡스
김신영
모름
이해찬
문재인
넬슨 만델라
김혜수
패리스 힐튼
박지선
송지효
옥택연
조승우
차은우
배두나
일론 머스크
조 바이든
홍석천
시진핑
강호동
모름
유재석
김선호
유아인
전현무
제니퍼 로페즈
싸이, 손연재, 류승룡
모름
버락 오바마
고학수
전지현
유아인
워렌 버핏

저우둥위
한소희
패리스 힐튼
소지섭
벤 맥켄지
모름
김희애
모름
소지섭
조국
모름
뷔
김희철
조지 클루니
장동건
제니
모름
마이클 잭슨
강승윤
모름
손흥민
박해미
리즈 위더스푼
판빙빙
타이거 우즈
조승연
김정은
이준석
정우성
송중기, 고민시
알베르트 아인슈타인


In [146]:
display_dataframe_with_images(df_2df)

image,name_100%
,알베르트 아인슈타인


In [149]:
df_3 = pd.DataFrame(columns=['image', 'name_100%'])

for i in range(1,122):
    names_100 = detect_with_gemini_url(f"image_{i}.jpeg", 100, False)
    image_data = f"<img src=\"https://storage.googleapis.com/{BUCKET}/human_images_2/image_{i}.jpeg\" width=\"40%\"/>"
    new_row = pd.DataFrame({'image': [image_data], 'name_100%': [names_100]})
    df_3 = pd.concat([df_3, new_row], ignore_index=False)

성룡
윤여정
레오나르도 디카프리오
홍현희
김수현
스티브 잡스
김신영
모름
이해찬
문재인


KeyboardInterrupt: 